### TOOLS

Models can request to call tools that perfrom tasks such as fetching data from a database,searching the web,or running code. tools are pairings of:
   1. A schema , including the name of the tool, a description, and / or argument definitions(often a JSOn schema)
   2. A fucntion or coroutine to execute

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-4.1-mini")
response = model.invoke("WHy do parrots talk ridiculously")
response

In [6]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get weather at a location"""
    return f"It is sunny at {location}"

model_with_tools = model.bind_tools([get_weather])

In [8]:
response = model_with_tools.invoke("What the weather like in Boston")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 49, 'total_tokens': 63, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5e793402c9', 'id': 'chatcmpl-DIqwypyvkLdjy9dIHVVfW6m4QZ2qr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ce608-14fd-7450-83df-69ae85c3ec09-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_75ehrsImkjhsaQZ8mwugOGvK', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 49, 'output_tokens': 14, 'total_tokens': 63, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Tool: get_weather
Args: {'

### TOOLS CALL EXECUTION

In [11]:
messages = [{"role":"user","content": "WHat's the weather in Boston"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny. Is there anything else you would like to know?


### Messages


Messages are the fundamental unit of context for models in langchain. They represent the input and output of models , carrying both the content and metadata needed to represent the state of a conversation when interacting with an llm. Messages are objects that contains :

-Role - Identifies the message type (e.g. System,user)
-Content - Represents the actual content of the message (like text , images,audio,documents, etc.)
-Metadata - Optional fields such as response information,message IDs , and token messages .

Langchain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [12]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1-mini")

In [13]:
model.invoke("Please tell what is artificial intelligence")

AIMessage(content='Artificial intelligence (AI) refers to the branch of computer science focused on creating systems or machines that can perform tasks typically requiring human intelligence. These tasks include learning, reasoning, problem-solving, understanding natural language, recognizing patterns, and making decisions. AI systems can range from simple algorithms to complex neural networks and are used in applications like virtual assistants, autonomous vehicles, recommendation systems, and more.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 13, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5e793402c9', 'id': 'chatcmpl-DIrwKyaby91OyQSk9vDYdxOt

### STRUCTURED OUPTPUT


Models can be requested to provide their response in a format matching a given schema. 
This is useful for ensuring the output can be easily parsed and used in subsequent processing. 
LangChain supports multiple schema types and methods for enforcing structured output.

### PYDANTIC

Pydantic models provide the richest feature set with field validation, descriptions, 
and nested structures.

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1-mini")